In [1]:
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks


def _bandpass(x, sr, lo=50.0, hi=900.0, order=3):
    nyq = 0.5 * sr
    b, a = butter(order, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a, x)


def _frame_signal(x, frame_len, hop):
    n = len(x)
    n_frames = 1 + max(0, (n - frame_len) // hop)
    frames = np.stack([x[i*hop:i*hop+frame_len] for i in range(n_frames)], axis=0)
    return frames


def _acf_pitch(frame, sr, fmin=60.0, fmax=400.0):
    """Return (f0_hz, strength) via normalized autocorrelation peak."""
    frame = frame - np.mean(frame)
    if np.max(np.abs(frame)) < 1e-6:
        return 0.0, 0.0

    # autocorrelation
    acf = np.correlate(frame, frame, mode="full")[len(frame)-1:]
    acf[0] = 0.0
    if np.max(acf) <= 0:
        return 0.0, 0.0
    acf = acf / (np.max(acf) + 1e-12)

    # lag range
    lag_min = int(sr / fmax)
    lag_max = int(sr / fmin)
    if lag_max >= len(acf):
        lag_max = len(acf) - 1
    if lag_min < 1 or lag_min >= lag_max:
        return 0.0, 0.0

    seg = acf[lag_min:lag_max]
    k = np.argmax(seg)
    lag = lag_min + k
    strength = seg[k]
    f0 = sr / lag if strength > 0 else 0.0
    return float(f0), float(strength)


def estimate_f0_track(x, sr, frame_ms=30, hop_ms=10, fmin=60, fmax=400,
                      voiced_thresh=0.35):
    frame_len = int(sr * frame_ms / 1000)
    hop = int(sr * hop_ms / 1000)
    frames = _frame_signal(x, frame_len, hop)

    f0 = np.zeros(len(frames))
    v = np.zeros(len(frames), dtype=bool)
    strength = np.zeros(len(frames))

    for i, fr in enumerate(frames):
        f0_i, s_i = _acf_pitch(fr, sr, fmin=fmin, fmax=fmax)
        strength[i] = s_i
        if s_i >= voiced_thresh and f0_i > 0:
            f0[i] = f0_i
            v[i] = True
        else:
            f0[i] = 0.0
            v[i] = False

    times = (np.arange(len(frames)) * hop + frame_len/2) / sr
    return times, f0, v, strength, frame_len, hop


def _interp_piecewise(t, t_points, y_points):
    """Linear interpolation of tier points."""
    return np.interp(t, t_points, y_points, left=y_points[0], right=y_points[-1])


def duration_tier_scale(t, tier_points):
    """
    tier_points: list of (time_seconds, scale_factor)
    scale_factor > 1 => slower, < 1 => faster
    """
    t_points = np.array([p[0] for p in tier_points], dtype=float)
    y_points = np.array([p[1] for p in tier_points], dtype=float)
    return _interp_piecewise(t, t_points, y_points)


def detect_pulses(x, sr, f0_times, f0_track, voiced, hop, search_ms=6):
    """
    Rough pitch-mark detection:
    - bandpass filter
    - in voiced frames, predict period from f0, then pick local peaks near expected instants
    Returns sample indices of pulses.
    """
    xf = _bandpass(x, sr, 50, 900)
    n = len(x)

    # Build a sample-wise f0 estimate by interpolating frame centers
    t_samp = np.arange(n) / sr
    f0_samp = np.interp(t_samp, f0_times, f0_track, left=0.0, right=0.0)
    v_samp = np.interp(t_samp, f0_times, voiced.astype(float), left=0.0, right=0.0) > 0.5

    pulses = []
    i = 0
    search = int(sr * search_ms / 1000)

    # Start at first voiced region
    while i < n and not v_samp[i]:
        i += 1

    while i < n:
        if not v_samp[i]:
            i += 1
            continue

        f0 = f0_samp[i]
        if f0 <= 0:
            i += 1
            continue

        period = int(sr / f0)
        if period < 10:
            i += 1
            continue

        # Find peak near i within +/- search
        lo = max(0, i - search)
        hi = min(n, i + search + 1)
        seg = xf[lo:hi]
        if len(seg) < 3:
            i += period
            continue

        # Peaks in magnitude (robust to polarity)
        peaks, _ = find_peaks(np.abs(seg))
        if len(peaks) == 0:
            # fallback: take max
            pk = int(np.argmax(np.abs(seg)))
        else:
            # pick the peak closest to center
            center = i - lo
            pk = peaks[np.argmin(np.abs(peaks - center))]
        pulse = lo + pk
        pulses.append(pulse)

        i = pulse + period  # jump ahead by one period

    # Remove duplicates and sort
    pulses = np.array(sorted(set(pulses)), dtype=int)
    return pulses


def td_psola_time_scale(x, sr, pulses, tier_points,
                        win_periods=2.0, crossfade_ms=5.0):
    """
    Voiced resynthesis: cut windows around pulses and OLA at new pulse times.
    Time-warp is defined by a duration tier scale s(t).
    """
    n = len(x)
    if len(pulses) < 3:
        return x.copy()

    # Estimate local period from pulse spacing
    dp = np.diff(pulses)
    dp = np.clip(dp, 20, int(sr/40))  # guard: 40 Hz floor for window sizing

    # Create mapping: original time -> new time via integral of scale
    # new_t(t) = ∫_0^t scale(u) du
    # We approximate numerically on pulse times.
    pulse_t = pulses / sr
    scale = duration_tier_scale(pulse_t, tier_points)

    # cumulative integral on pulses
    dt = np.diff(pulse_t, prepend=pulse_t[0])
    new_pulse_t = np.cumsum(scale * dt)
    # normalize so starts at 0
    new_pulse_t -= new_pulse_t[0]
    new_pulses = np.round(new_pulse_t * sr).astype(int)

    out_len = int(new_pulses[-1] + (win_periods * np.max(dp)) + 2)
    y = np.zeros(out_len, dtype=float)

    # overlap-add
    for k, p in enumerate(pulses):
        # local period estimate
        if k == 0:
            T = dp[0]
        elif k >= len(dp):
            T = dp[-1]
        else:
            T = dp[k-1]
        half = int(win_periods * T / 2)
        if half < 16:
            half = 16

        a = max(0, p - half)
        b = min(n, p + half)
        seg = x[a:b].copy()

        # Hann window for smooth OLA
        w = np.hanning(len(seg))
        seg *= w

        q = new_pulses[k]
        ya = max(0, q - (p - a))
        yb = ya + len(seg)
        if yb > len(y):
            y = np.pad(y, (0, yb - len(y)))
        y[ya:yb] += seg

    # Optional gentle normalization
    m = np.max(np.abs(y)) + 1e-12
    y = y / m * (np.max(np.abs(x)) + 1e-12)

    return y


# --- Example usage ---
if __name__ == "__main__":
    import soundfile as sf  # pip install soundfile

    x, sr = sf.read("input.wav")
    if x.ndim > 1:
        x = x.mean(axis=1)

    # 1) F0/voicing (for pulse finding)
    t, f0, voiced, strength, frame_len, hop = estimate_f0_track(
        x, sr, fmin=60, fmax=400, voiced_thresh=0.35
    )

    # 2) Pulse detection (pitch marks)
    pulses = detect_pulses(x, sr, t, f0, voiced, hop)

    # 3) Duration tier: speed up globally by 10% => scale=0.90
    tier = [(0.0, 0.90), (len(x)/sr, 0.90)]

    # 4) Voiced PSOLA time scaling
    y = td_psola_time_scale(x, sr, pulses, tier)

    sf.write("output_psola.wav", y, sr)
